<a href="https://colab.research.google.com/github/kimcanal/robo-advisor-rl/blob/master/colab_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RL 모듈 — 최종 검증 실행 (Colab)

지금까지 찾은 버그 3개(거래비용 2배 계산, 무위험이자율 하드코딩, MACD look-ahead bias)와 VecNormalize가 **전부 한꺼번에 적용된 상태로 Walk-Forward 검증을 다시 돌리는 노트북**입니다. 기존 발표자료의 수치는 이 수정들이 부분적으로만 반영된 상태였어서, 여기 결과가 '진짜 최종' 숫자입니다.

**실행 방법**: 위 메뉴 `런타임 > 모두 실행` (GPU 필요 없음, CPU 런타임으로 충분 — 작은 신경망이라 GPU가 오히려 이득 없음)

**예상 소요시간**: 약 25~35분 (셀 3번)

**주의**: Colab 무료 플랜은 브라우저 탭과 상호작용이 90분 이상 없으면 런타임이 끊길 수 있습니다. 가끔 탭을 확인해주세요.

In [1]:
# 셀 1: 레포 클론 + 의존성 설치
!git clone https://github.com/kimcanal/robo-advisor-rl.git
%cd robo-advisor-rl
!pip install -q -r rl/requirements.txt

In [2]:
# 셀 2: 실제 ETF 10종 데이터 받기 (라이선스상 레포엔 CSV가 없어서 직접 받아야 함)
!python -m rl.data.fetch_real_data

[fetch] SPY: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/SPY.csv
[fetch] QQQ: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/QQQ.csv
[fetch] IWM: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/IWM.csv
[fetch] EFA: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/EFA.csv
[fetch] EEM: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/EEM.csv
[fetch] AGG: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/AGG.csv
[fetch] TLT: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/TLT.csv
[fetch] HYG: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/HYG.csv
[fetch] GLD: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/GLD.csv
[fetch] VNQ: 1948행, 2019-01-02~2026-10-01 -> /content/robo-advisor-rl/rl/data/raw/VNQ.csv


In [9]:
# 셀 3: 최종 Walk-Forward 검증 (2개 윈도우 x 보상함수 3종 x 시드 5개, 전체 버그 수정 반영)
# 약 25~35분 소요
!python -m rl.walk_forward --real --timesteps 120000 --seeds 0,1,2,3,4

2026-10-02 12:03:18.827442: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Cookie/crumb fetch failed (ImpersonateError), continuing without crumb
Failed to get ticker 'BIL' reason: Impersonating chrome150 is not supported
Cookie/crumb fetch failed (ImpersonateError), continuing without crumb

1 Failed download:
['BIL']: ImpersonateError('Impersonating chrome150 is not supported')
[walk_forward] ri

In [10]:
import os
import glob

# rl 디렉토리 내부에서 생성된 폴더 및 파일 구조를 탐색합니다.
print("=== rl 디렉토리 구조 탐색 ===")
for root, dirs, files in os.walk('rl'):
    # 불필요한 pycache나 __pycache__는 제외하고 출력합니다.
    if '__pycache__' in root:
        continue

    level = root.replace('rl', '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}[D] {os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        if not f.endswith('.pyc'):
            print(f"{subindent}- {f}")

=== rl 디렉토리 구조 탐색 ===
[D] rl/
    - __init__.py
    - experiments.py
    - walk_forward.py
    - shap_explain.py
    - backtest.py
    - pipeline.py
    - vecnorm.py
    - regime.py
    - rewards.py
    - README.md
    - config.py
    - run_demo.py
    - requirements.txt
    - stats_tests.py
    - mvo.py
    - features.py
    - train.py
    - riskfree.py
    [D] outputs/
        [D] walk_forward/
    [D] tests/
        - __init__.py
        - test_rewards.py
        - test_backtest.py
        - test_env.py
    [D] env/
        - __init__.py
        - portfolio_env.py
    [D] data/
        - __init__.py
        - loader.py
        - dummy_data.py
        - fetch_real_data.py
        [D] raw/
            - EFA.csv
            - HYG.csv
            - EEM.csv
            - QQQ.csv
            - IWM.csv
            - SPY.csv
            - GLD.csv
            - VNQ.csv
            - AGG.csv
            - TLT.csv


In [7]:
# 셀 4: 결과 확인
import pandas as pd

results = pd.read_csv('rl/outputs/walk_forward/walk_forward_results.csv')
print('=== 윈도우별 요약 ===')
display(results[['window', 'strategy', 'seed', 'cumulative_return', 'sharpe', 'mdd']])

daily = pd.read_csv('rl/outputs/walk_forward/walk_forward_daily.csv')
print(f"\n일별 레코드 수: {len(daily)}")
print(daily['regime'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'rl/outputs/walk_forward/walk_forward_results.csv'

In [ ]:
# 셀 5 (선택): 결과 CSV를 로컬로 다운로드
from google.colab import files
files.download('rl/outputs/walk_forward/walk_forward_results.csv')
files.download('rl/outputs/walk_forward/walk_forward_daily.csv')